In [ ]:
import torch
import numpy as np
import torch.nn as nn
import numpy as np
import math
import pandas as pd 
from tensorflow.keras.preprocessing.text import Tokenizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset


Token Embedding Class

In [ ]:


class Embedding(nn.Module):

    def __init__(self, vocab_size, d_model):
        super(Embedding, self).__init__()

        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
            
    def forward(self, x):
        return self.embedding(x) 
    



Position_Encoding Class

In [ ]:


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        # Create a matrix of [max_len, d_model] representing positions
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Calculate the division term using the log-space trick for numerical stability
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Fill the even indices with sine and odd with cosine
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add a batch dimension: (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        
        # register_buffer ensures pe is moved to GPU with the model but not trained
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        # Add the positional encoding to the embeddings
        x = x + self.pe[:, :x.size(1), :]
        return x




MultiHead_Attention CLass

In [ ]:
class multihead_attention(torch.nn.Module):
    
    def __init__(self, d_model, num_heads):
        super(multihead_attention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        assert self.head_dim * num_heads == d_model, "d_model must be divisible by num_heads"
        
        self.W_q = torch.nn.Linear(d_model, d_model)
        self.W_k = torch.nn.Linear(d_model, d_model)
        self.W_v = torch.nn.Linear(d_model, d_model)
        self.W_o = torch.nn.Linear(d_model, d_model)
    

    
    def forward(self, x):
        batch_size, seq_length, d_model = x.size()
        
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        Q = Q.view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.head_dim)

        # Keep mask on same device as scores to avoid CPU/GPU mismatch.
        mask = torch.triu(
            torch.ones(seq_length, seq_length, device=x.device), diagonal=1
        ).bool()
        
        scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        
        attn_weights = torch.nn.functional.softmax(scores, dim=-1)
        
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, d_model)
        
        output = self.W_o(attn_output)
        
        return output 
    



LayerNormalization Class

In [ ]:
class layernormalization(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super(layernormalization, self).__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps
       
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        normalized_x = (x - mean) / (std + self.eps)
        output = self.gamma * normalized_x + self.beta
        return output


Feedforward Class

In [ ]:
class feedforward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(feedforward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        
    def forward(self, x):
        x = torch.relu(self.linear1(x))
        output = self.linear2(x)
        return output

TRAINING

In [ ]:


with open("C:\\Code\\Jupter\\ML\\Project\\GPT_From_scratch\\hamlet.txt", "r", encoding="utf-8") as f:
    text = f.read()   # read entire file as string

words = word_tokenize(text)
word=[w for w in words if w not in ['.',',','!','?',';',':','(',')','[',']','{','}','"',"'",'-'] and w.isalpha() and len(w)>1]


tokenizer = Tokenizer()
tokenizer.fit_on_texts(word)
sequences = tokenizer.texts_to_sequences(word)
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary Size:", vocab_size)



In [ ]:
label=tokenizer.word_index
y={v: k for k, v in tokenizer.word_index.items()}


In [ ]:
input=[]

for i in text.split('\n'):
    load=tokenizer.texts_to_sequences([i])[0]
    
    for j in range(1, len(load)):
        input.append(load[:j+1])


In [ ]:




max_len = max(len(seq) for seq in input)

padded = torch.zeros(len(input), max_len, dtype=torch.long)

for i, seq in enumerate(input):
  
    padded[i, -len(seq):] = torch.tensor(seq)



In [ ]:
x,y =padded[:,:-1], padded[:,-1]


In [ ]:

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [ ]:
vocab_size

In [ ]:

x_train_tensor = torch.tensor(x_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)   

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

#Testing data preparation
x_test_tensor = torch.tensor(x_test, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

test_dataset = TensorDataset(x_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


class transformer(nn.Module):

    def __init__(self, d_model, vocab_size):

        super(transformer, self).__init__()
        
    

        self.embedding = Embedding(vocab_size, d_model)

        self.pos_encoding = PositionalEncoding(d_model)

        self.attention = multihead_attention(d_model, num_heads=2)

        self.norm1 = layernormalization(d_model)

        self.feedforward = feedforward(d_model, d_ff=48)

        self.norm2 = layernormalization(d_model)

        self.linear = nn.Linear(d_model, vocab_size)


    def forward(self, x):

        output = self.embedding(x)

        output = self.pos_encoding(output)
        res1=output 
        output = self.attention(output)

        # Add & Norm
        output = output + res1 
        output = self.norm1(output)

        output = self.feedforward(output)

        output = self.norm2(output)

        output = self.linear(output)

        return output
        
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

criterion = nn.CrossEntropyLoss()

model = transformer(d_model=6, vocab_size=vocab_size).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epochs in range(10000):
    model.train()
    running_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        output = model(batch_x)
        output = output[:, -1, :]   # take last token prediction
        loss = criterion(output, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        

        preds = torch.argmax(output, dim=-1)

        running_loss += loss.item()
        
    
    print(f"Epoch [{epochs+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# ==========================
# Evaluation
# ==========================
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        predicted = outputs[:, -1, :] 
        predicted = torch.argmax(predicted, dim=-1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\nTest Accuracy: {accuracy:.2f}%")
